In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table(
    "hackathon.shared_datasets.fraud_busters_train_joined"
)

print(df.count())
print(len(df.columns))


590540
434


In [0]:

missing_indicator_cols = [
    "DeviceType",
    "DeviceInfo",
    "id_01",
    "id_02",
    "id_30",
    "id_31"
]

for col_name in missing_indicator_cols:
    df = df.withColumn(
        f"{col_name}_missing",
        F.when(
            F.col(col_name).isNull(),
            1
        ).otherwise(0)
    )


In [0]:

df = (
    df
    .withColumn(
        "hour_of_day",
        ((F.col("TransactionDT") % 86400) / 3600).cast("int")
    )
    .withColumn(
        "day_number",
        (F.col("TransactionDT") / 86400).cast("int")
    )
)


In [0]:

df = df.withColumn(
    "day_mod_7",
    F.col("day_number") % 7
)


In [0]:

df = df.withColumn(
    "log_transaction_amt",
    F.log1p(F.col("TransactionAmt"))
)


In [0]:

p99 = (
    df
    .selectExpr(
        "percentile_approx(TransactionAmt,0.99) as p99"
    )
    .collect()[0]["p99"]
)

df = df.withColumn(
    "high_value_txn",
    F.when(
        F.col("TransactionAmt") > p99,
        1
    ).otherwise(0)
)



In [0]:

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(
          "hackathon.shared_datasets.fraud_busters_features"
      )
)
print(df)
print("✅ Feature table saved")


DataFrame[TransactionID: bigint, isFraud: bigint, TransactionDT: bigint, TransactionAmt: double, ProductCD: string, card1: bigint, card2: double, card3: double, card4: string, card5: double, card6: string, addr1: double, addr2: double, dist1: double, dist2: double, P_emaildomain: string, R_emaildomain: string, C1: double, C2: double, C3: double, C4: double, C5: double, C6: double, C7: double, C8: double, C9: double, C10: double, C11: double, C12: double, C13: double, C14: double, D1: double, D2: double, D3: double, D4: double, D5: double, D6: double, D7: double, D8: double, D9: double, D10: double, D11: double, D12: double, D13: double, D14: double, D15: double, M1: string, M2: string, M3: string, M4: string, M5: string, M6: string, M7: string, M8: string, M9: string, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: doub